In [ ]:
X_train, y_train = generate_date(7, 1)

In [ ]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF



# Convert inputs to tuples for quick lookup
input_tuples = [tuple(row) for row in X_train]
output_dict = dict(zip(input_tuples, y_train))

# Parameters
noise_assumption = 1e-10
rbf_lengthscale = 1
beta = 1.4
input_dim = X_train.shape[1]
num_queries = 500  # how many BO steps you want to perform

# Initialize dataset for BO (start from scratch or with some data)
X_BO = []
Y_BO = []

kernel = RBF(length_scale=rbf_lengthscale, length_scale_bounds='fixed')
model = GaussianProcessRegressor(kernel=kernel, alpha=noise_assumption)

In [ ]:
model.fit(X_train, y_train)

candidate_points = np.random.rand(1000, X_train.shape[1])
candidate_filtered = [
        pt for pt in candidate_points
        if not any(np.allclose(pt, xq) for xq in X_BO)
    ]
candidate_filtered = np.array(candidate_filtered)
post_mean, post_std = model.predict(candidate_filtered, return_std=True)
acquisition_function = post_mean + beta * post_std
x_next = candidate_filtered[np.argmax(acquisition_function)]



In [ ]:
candidate_points = np.random.rand(1000, X_train.shape[1])
candidate_filtered = [
        pt for pt in candidate_points
        if not any(np.allclose(pt, xq) for xq in X_BO)
    ]
candidate_filtered = np.array(candidate_filtered)
post_mean, post_std = model.predict(candidate_filtered, return_std=True)
acquisition_function = post_mean + beta * post_std
x_next = candidate_filtered[np.argmax(acquisition_function)]

In [ ]:
def select_next_point(model, candidate_points, X_BO, beta, eta, acquisition='ucb'):
    post_mean, post_std = model.predict(candidate_points, return_std=True)
    y_max = max(Y_BO) if Y_BO else -np.inf

    if acquisition == 'ucb':
        acquisition_values = acquisition_ucb(post_mean, post_std, beta)
    elif acquisition == 'pi':
        acquisition_values = acquisition_pi(post_mean, post_std, y_max, eta)
    else:
        raise ValueError("Acquisition function must be 'ucb' or 'pi'")

    # Filter out points already queried
    candidate_filtered = [
        pt for pt in candidate_points
        if not any(np.allclose(pt, xq) for xq in X_BO)
    ]

    if len(candidate_filtered) == 0:
        print("No new candidates left to query.")
        return None

    post_mean_filt, post_std_filt = model.predict(candidate_filtered, return_std=True)
    if acquisition == 'ucb':
        acquisition_filt = acquisition_ucb(post_mean_filt, post_std_filt, beta)
    else:
        acquisition_filt = acquisition_pi(post_mean_filt, post_std_filt, y_max, eta)

    idx_best = np.argmax(acquisition_filt)
    return candidate_filtered[idx_best]

In [ ]:
beta = 0.9;
select_next_point(model, candidate_points, X_BO,beta, eta, acquisition='ucb')

X_BO.append(x_next)
Y_BO.append(y_next)

best_idx = np.argmax(Y_BO)
arrayubcb500 = X_BO[best_idx]
print("\nBest found input:", X_BO[best_idx])
print("Best output value:", Y_BO[best_idx])


In [ ]:
x_next = select_next_point(model, candidate_points, X_BO, beta, eta, acquisition='ucb')
print("Next selected point:", x_next)

In [ ]:
eta = 9
x_next = select_next_point(model, candidate_points, X_BO,90, eta, 'ucb')
if tuple(x_next) in output_dict:
    y_next = output_dict[tuple(x_next)]
    print(f"Query {len(X_BO)+1}: Using existing output for {x_next}, output = {y_next:.4f}")
else:
   
    y_next, _ = model.predict(x_next.reshape(1, -1), return_std=True)
    y_next = y_next.item()
    print(f"Query {len(X_BO)+1}: No true output available; using model prediction {y_next:.4f}")
best_idx = np.argmax(Y_BO)

X_BO.append(x_next)
Y_BO.append(y_next)
print("\nBest found input:", X_BO[best_idx])
print("Best output value:", Y_BO[best_idx])


eta = 0.0
select_next_point(model, candidate_points, X_BO,beta, eta, 'pi')

X_BO.append(x_next)
Y_BO.append(y_next)
y_next, _ = model.predict(x_next.reshape(1, -1), return_std=True)
y_next = y_next[0]  # scalar

best_idx = np.argmax(Y_BO)
arraypieta0 = X_BO[best_idx]
print("\nBest found input:", X_BO[best_idx])
print("Best output value:", Y_BO[best_idx])
